# Resultados reprodutíveis

Refaça um lote simulado com a mesma semente. Preveja o que muda se houver uma chamada adicional para gerar um número. Retome [ambiente](https://devrgs.github.io/Aprendendo-linguagens/#/python/projetos/ambiente-e-pacotes), [JSON](https://devrgs.github.io/Aprendendo-linguagens/#/python/arquivos/json) e [conjuntos](https://devrgs.github.io/Aprendendo-linguagens/#/python/registros/conjuntos).

Defina uma ordem estável para entradas e saídas. `random.Random(semente)` cria um gerador pseudoaleatório local. Chamadas avançam seu estado, então anote a semente, o método e a sequência de operações. Registre a versão do ambiente, os parâmetros e a proveniência, ou seja, de onde vieram os dados e como foram preparados. Esses dados permitem investigar diferenças entre execuções.

O código usa entradas pequenas no caderno e no kit. Abra o JSON de E3 na pasta de resultados para conferir versões e contagens. O que precisa ser igual antes de comparar duas execuções? Depois, veja [vetores](https://devrgs.github.io/Aprendendo-linguagens/#/python/matematica/vetores).

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-reprodutibilidade-', dir=_raiz))


## E1 · Preveja e execute

Crie dois objetos Random com a mesma semente e faça três chamadas random em cada um. Compare as sequências.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
from random import Random

a, b = Random(7), Random(7)
primeira = [a.random() for _ in range(3)]
segunda = [b.random() for _ in range(3)]
print(primeira == segunda, len(primeira))


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
True 3
```

Os estados iniciais e a sequência de chamadas coincidem, por isso os valores coincidem. Cada objeto mantém seu estado separado.


## E2 · Altere ou depure

Use a mesma semente nos dois geradores, mas faça uma chamada a mais no segundo antes de comparar. Verifique se seu próximo valor coincide com o segundo valor do primeiro gerador.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
from random import Random

a, b = Random(7), Random(7)
primeiro = a.random()
segundo = a.random()
b.random()
deslocado = b.random()
print(primeiro == deslocado)
print(segundo == deslocado)


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
False
True
```

A chamada extra consome o primeiro valor. Ela desloca o ponto da sequência, como mostra a comparação com o segundo.


## E3 · Resolva

Produza `auditoria.json` com IDs únicos ordenados, contagens, semente, versão Python e origem do lote. Para ["b", "a", "b"], faça dois resumos iguais e confirme contagens 3 e 2. Para [] espere 0 IDs e zero contagens.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
import json
import sys
from pathlib import Path

def resumir(ids, semente):
    unicos = sorted(set(ids))
    return {
        "ids": unicos,
        "quantidade": len(ids),
        "unicos": len(unicos),
        "semente": semente,
        "python": ".".join(map(str, sys.version_info[:3])),
        "origem": "lote fictício da atividade",
    }

entrada = ["b", "a", "b"]
primeiro = resumir(entrada, 7)
caminho = Path("auditoria.json")
caminho.write_text(json.dumps(primeiro, ensure_ascii=False, sort_keys=True),
                   encoding="utf-8")
recuperado = json.loads(caminho.read_text(encoding="utf-8"))
print(recuperado["ids"], recuperado["quantidade"], recuperado["unicos"])
print(recuperado == resumir(entrada, 7))
vazio = resumir([], 7)
print(vazio["ids"], vazio["quantidade"], vazio["unicos"])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
['a', 'b'] 3 2
True
[] 0 0
```

Ordenar estabiliza a saída do conjunto. sort_keys organiza o JSON por chave. O resumo guarda a semente como parâmetro de uma eventual simulação. Nenhum sorteio é feito nesta auditoria. A entrada vazia continua reproduzível, com ambas as contagens em zero.
